# Pacientes vistos en el período del reporte de citas

Combina el reporte de citas de `../Data/Appointments` con las listas de pacientes de `../Data/Patients`
y exporta un CSV con un registro por paciente: `Patients seen from MM.DD.YYYY-MM.DD.YYYY.csv`.

In [ ]:
import glob
import os
import re

import pandas as pd

In [ ]:
APPOINTMENTS_DIR = '../Data/Appointments'
PATIENTS_DIR = '../Data/Patients'
OUTPUT_PREFIX = 'Patients seen from'
ENCODING = 'latin-1'

STATUSES_TO_DROP = ['Cancelled', 'No Show', 'Pending']
APPOINTMENT_COLUMNS_TO_DROP = ['MobilePhone', 'HomePhone', 'OfficePhone', 'AppointmentStatus', 'Copay', 'Eligibility']

APPOINTMENT_STATUS_PRIORITY = ['Seen', 'In room', 'In lobby']
APPOINTMENT_TYPE_PRIORITY = [
    'Urgent Visit', 'Podiatry', 'New Patient Visit', 'Nutrition', 'Video Visit',
    'Wellness Visit', 'Wellness Exam', 'Immigration Physical', 'Physical Exam', 'X-Ray',
    'Upper Sono', 'Lower Sono', 'Care Coordination', 'Allergy Testing', 'ABI Exam',
    'DOT Physical', 'Follow-Up Visit', 'XRAY Result Follow UP', 'Nursing Only',
    'TCM Remote Care', 'Remote Monitoring', 'RPM Delivery',
]

PATIENT_COLUMNS_TO_DROP = [
    'Ethnicity', 'Preferred Language', 'Ethnicity / Preferred Language Effective Date',
    'Race', 'Race Effective Date', 'Smoking Status', 'Smoking Status Effective Date',
    'Preferred Method of Communication', 'Patient Middle Name',
]

## Funciones auxiliares

In [ ]:
def require_columns(df, columns, source):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"Faltan columnas en {source}: {missing}")


def drop_columns(df, columns, source):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        print(f"Aviso: columnas no encontradas en {source} (se ignoran): {missing}")
    return df.drop(columns=columns, errors='ignore')


def match_key(names, dobs):
    """Clave para cruzar pacientes: nombre sin espacios extra ni mayúsculas + DOB como fecha.
    Solo se usa para comparar; los datos originales no se modifican."""
    normalized_names = names.fillna('').astype(str).str.split().str.join(' ').str.lower()
    return pd.MultiIndex.from_arrays([normalized_names, pd.to_datetime(dobs, errors='coerce')])


def rank(values, priority):
    """Posición de cada valor en la lista de prioridad; los que no están quedan al final."""
    positions = {value.lower(): i for i, value in enumerate(priority)}
    return values.fillna('').str.strip().str.lower().map(positions).fillna(len(priority))


def proper_name(name):
    """Como PROPER de Excel: solo letras, espacios, guiones y apóstrofes.
    Los espacios junto a un guion se eliminan (Mary- Ann -> Mary-Ann).
    Guiones y apóstrofes al inicio o final de una palabra se eliminan,
    salvo apóstrofes que encierran la palabra completa ('Ed')."""
    if pd.isna(name):
        return pd.NA
    kept = ''.join(ch for ch in str(name) if ch.isalpha() or ch.isspace() or ch in "-'")
    kept = re.sub(r'\s*-\s*', '-', kept.strip())
    words = []
    for word in kept.split():
        quoted = len(word) > 2 and word[0] == "'" and word[-1] == "'"
        core = (word[1:-1] if quoted else word).strip("-'")
        if not core:
            continue
        # Mayúscula después de cualquier carácter que no sea letra, como PROPER (O'Donnell, Price-White)
        titled = ''.join(
            ch.upper() if i == 0 or not core[i - 1].isalpha() else ch.lower()
            for i, ch in enumerate(core)
        )
        words.append(f"'{titled}'" if quoted else titled)
    return ' '.join(words) or pd.NA


def clean_address_part(part, state=None):
    """Sin espacios extra, empezando y terminando en letra o número, y con mayúscula
    solo en la primera letra de cada palabra (también después de un guion).
    Excepciones en mayúsculas: apartamentos con números y letras (1A, 2FL) y el
    estado del paciente si aparece escrito en la dirección (NY). Los ordinales
    (1st, 94th) quedan en minúscula."""
    if pd.isna(part):
        return ''
    cleaned = re.sub(r'^[^0-9A-Za-z]+|[^0-9A-Za-z]+$', '', str(part).strip())
    cleaned = ''.join(
        ch.upper() if i == 0 or cleaned[i - 1] in ' -' else ch.lower()
        for i, ch in enumerate(cleaned)
    )

    def fix_word(match):
        word = match.group(0)
        if re.fullmatch(r'\d+(st|nd|rd|th)', word):
            return word
        return word.upper()

    cleaned = re.sub(r'\b(?=[A-Za-z]*\d)(?=\d*[A-Za-z])[0-9A-Za-z]+\b', fix_word, cleaned)
    if isinstance(state, str) and state.strip():
        cleaned = re.sub(rf'\b{re.escape(state.strip())}\b', state.strip().upper(), cleaned, flags=re.IGNORECASE)
    return cleaned


def format_phone(phone):
    """(###) ###-#### si tiene exactamente 10 dígitos; si no, se descarta."""
    if pd.isna(phone):
        return pd.NA
    digits = re.sub(r'\D', '', str(phone))
    if len(digits) != 10:
        return pd.NA
    return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

## 1. Leer el reporte de citas

In [ ]:
# El CSV exportado por este notebook también se guarda en esta carpeta, así que se excluye
appointment_files = [
    f for f in glob.glob(os.path.join(APPOINTMENTS_DIR, '*.csv'))
    if not os.path.basename(f).startswith(OUTPUT_PREFIX)
]
if len(appointment_files) != 1:
    raise FileNotFoundError(
        f"Se esperaba exactamente 1 reporte de citas en {APPOINTMENTS_DIR}, se encontraron "
        f"{len(appointment_files)}: {[os.path.basename(f) for f in appointment_files]}"
    )
appointment_file = appointment_files[0]

df_appointments = pd.read_csv(appointment_file, encoding=ENCODING, dtype=str, keep_default_na=False, na_values=[''])
require_columns(
    df_appointments,
    ['AppointmentTime', 'Patient', 'DOB', 'AppointmentType', 'AppointmentStatus', 'SeenBy', 'Facility'],
    os.path.basename(appointment_file),
)

df_appointments['_time'] = pd.to_datetime(df_appointments['AppointmentTime'], format='%m/%d/%Y %I:%M %p', errors='coerce')
df_appointments['_dob'] = pd.to_datetime(df_appointments['DOB'], format='%m/%d/%Y', errors='coerce')

bad_times = df_appointments[df_appointments['_time'].isna()]
if not bad_times.empty:
    print(f"Aviso: {len(bad_times)} citas con AppointmentTime inválido (se eliminan):")
    print(bad_times[['Patient', 'DOB', 'AppointmentTime']].to_string())
    df_appointments = df_appointments[df_appointments['_time'].notna()]

# Fechas del reporte original para el nombre del archivo final
report_start = df_appointments['_time'].min()
report_end = df_appointments['_time'].max()

print(f"{os.path.basename(appointment_file)}: {len(df_appointments)} citas, "
      f"del {report_start:%m/%d/%Y} al {report_end:%m/%d/%Y}")

## 2. Eliminar citas Cancelled, No Show y Pending

In [ ]:
status = df_appointments['AppointmentStatus'].fillna('').str.strip().str.lower()
df_appointments = df_appointments[~status.isin([s.lower() for s in STATUSES_TO_DROP])]
print(f"Citas restantes: {len(df_appointments)}")
df_appointments['AppointmentStatus'].value_counts()

## 3. Eliminar pacientes menores de edad

In [ ]:
bad_dobs = df_appointments[df_appointments['_dob'].isna()]
if not bad_dobs.empty:
    print(f"Aviso: {len(bad_dobs)} citas con DOB inválido o vacío (se eliminan):")
    print(bad_dobs[['Patient', 'DOB', 'AppointmentTime']].to_string())

adult_cutoff = pd.Timestamp.today().normalize() - pd.DateOffset(years=18)
df_appointments = df_appointments[df_appointments['_dob'] <= adult_cutoff]
print(f"Citas de adultos: {len(df_appointments)}")

## 4–6. Quedarse con la cita más reciente de cada paciente

Desempate a la misma fecha y hora: primero `AppointmentStatus` (Seen > In room > In lobby),
luego `AppointmentType` según la lista de prioridad. Si son idénticas, se deja cualquiera.

`AppointmentType` y `SeenBy` no se eliminan porque van en el archivo final.

In [ ]:
df_appointments['_key_name'] = match_key(df_appointments['Patient'], df_appointments['DOB']).get_level_values(0)
df_appointments['_status_rank'] = rank(df_appointments['AppointmentStatus'], APPOINTMENT_STATUS_PRIORITY)
df_appointments['_type_rank'] = rank(df_appointments['AppointmentType'], APPOINTMENT_TYPE_PRIORITY)

unknown_types = sorted(set(df_appointments.loc[df_appointments['_type_rank'] == len(APPOINTMENT_TYPE_PRIORITY), 'AppointmentType'].dropna()))
if unknown_types:
    print(f"Aviso: AppointmentType sin prioridad definida (quedan al final): {unknown_types}")

df_latest_appointments = (
    df_appointments
    .sort_values(['_time', '_status_rank', '_type_rank'], ascending=[False, True, True])
    .drop_duplicates(subset=['_key_name', '_dob'], keep='first')
    .drop(columns=['_status_rank', '_type_rank'])
)
df_latest_appointments = drop_columns(df_latest_appointments, APPOINTMENT_COLUMNS_TO_DROP, 'el reporte de citas')

print(f"Pacientes únicos en citas: {len(df_latest_appointments)}")
df_latest_appointments.drop(columns=['_time', '_dob', '_key_name']).head()

## 7. Combinar las listas de pacientes

In [ ]:
patient_files = sorted(glob.glob(os.path.join(PATIENTS_DIR, '*.csv')))
if not patient_files:
    raise FileNotFoundError(f"No hay archivos .csv en {PATIENTS_DIR}")

patient_dfs = []
for file in patient_files:
    try:
        df = pd.read_csv(file, encoding=ENCODING, dtype=str, keep_default_na=False, na_values=[''])
    except Exception as e:
        raise RuntimeError(f"No se pudo leer {file}: {e}") from e
    require_columns(
        df,
        ['Patient First Name', 'Patient Last Name', 'Patient Identifier', 'Date of Birth', 'Age', 'Sex',
         'Address Line 1', 'Address Line 2', 'City', 'State', 'Postal Code', 'Most Recent Visit Date',
         'Home Phone', 'Mobile Phone', 'Office Phone', 'Email Address', 'Active', 'Fake'],
        os.path.basename(file),
    )
    print(f"{os.path.basename(file)}: {len(df)} pacientes")
    patient_dfs.append(df)

df_patients = pd.concat(patient_dfs, ignore_index=True)
print(f"Total combinado: {len(df_patients)}")

## 8. Eliminar columnas de pacientes

In [ ]:
df_patients = drop_columns(df_patients, PATIENT_COLUMNS_TO_DROP, 'la lista de pacientes')
list(df_patients.columns)

## 9. ZIP Codes: solo los primeros 5 dígitos, como entero (sin ceros a la izquierda)

In [ ]:
zip_digits = df_patients['Postal Code'].fillna('').str.replace(r'\D', '', regex=True).str[:5]
df_patients['Postal Code'] = pd.to_numeric(zip_digits.replace('', pd.NA), errors='coerce').astype('Int64')
df_patients['Postal Code'].head()

## 10. Quedarse solo con los pacientes que están en el reporte de citas

In [ ]:
patient_keys = match_key(df_patients['Patient First Name'] + ' ' + df_patients['Patient Last Name'], df_patients['Date of Birth'])
appointment_keys = pd.MultiIndex.from_arrays([df_latest_appointments['_key_name'], df_latest_appointments['_dob']])

df_patients['_key_name'] = patient_keys.get_level_values(0)
df_patients['_dob'] = patient_keys.get_level_values(1)
df_patients = df_patients[patient_keys.isin(appointment_keys)]

not_in_patients = df_latest_appointments[~appointment_keys.isin(patient_keys)]
if not not_in_patients.empty:
    print(f"ERROR: {len(not_in_patients)} paciente(s) del reporte de citas no están en la lista de pacientes:")
    print(not_in_patients[['Patient', 'DOB', 'AppointmentTime']].to_string(index=False))

print(f"Registros de pacientes encontrados (con duplicados): {len(df_patients)}")

## 11. Eliminar pacientes duplicados

Solo entre registros con el mismo nombre y DOB, en este orden de prioridad:
1. `Fake` = No
2. `Active` = Yes
3. `Patient Identifier` que no contenga "dup"
4. `Most Recent Visit Date` más reciente
5. `Patient Identifier` más corto (sin contar espacios)
6. Si sigue el empate, cualquiera

In [ ]:
df_patients = df_patients.assign(
    _not_fake=df_patients['Fake'].fillna('').str.strip().str.lower().eq('no'),
    _active=df_patients['Active'].fillna('').str.strip().str.lower().eq('yes'),
    _no_dup=~df_patients['Patient Identifier'].fillna('').str.contains('dup', case=False),
    _last_visit=pd.to_datetime(df_patients['Most Recent Visit Date'], errors='coerce'),
    _id_length=df_patients['Patient Identifier'].fillna('').str.replace(r'\s', '', regex=True).str.len(),
)
df_unique_patients = (
    df_patients
    .sort_values(
        ['_not_fake', '_active', '_no_dup', '_last_visit', '_id_length'],
        ascending=[False, False, False, False, True],
        na_position='last',
    )
    .drop_duplicates(subset=['_key_name', '_dob'], keep='first')
    .drop(columns=['_not_fake', '_active', '_no_dup', '_last_visit', '_id_length'])
)
print(f"Pacientes únicos: {len(df_unique_patients)} | Pacientes en citas: {len(df_latest_appointments)} "
      f"| No encontrados: {len(not_in_patients)}")

## 12. Combinar pacientes y citas

In [ ]:
df_merged = df_unique_patients.merge(
    df_latest_appointments[['_key_name', '_dob', '_time', 'AppointmentType', 'SeenBy', 'Facility']],
    on=['_key_name', '_dob'],
    how='inner',
    validate='one_to_one',
)
if len(df_merged) != len(df_latest_appointments) - len(not_in_patients):
    raise ValueError(
        f"El cruce dio {len(df_merged)} filas, se esperaban {len(df_latest_appointments) - len(not_in_patients)}"
    )

line_1 = pd.Series(
    [clean_address_part(a, s) for a, s in zip(df_merged['Address Line 1'], df_merged['State'])], index=df_merged.index
)
line_2 = pd.Series(
    [clean_address_part(a, s) for a, s in zip(df_merged['Address Line 2'], df_merged['State'])], index=df_merged.index
)
address = line_1.where(line_2 == '', line_1 + ', ' + line_2)

# Teléfonos corridos a la izquierda: el primero válido en Phone 1, el segundo en Phone 2, etc.
phones = df_merged[['Home Phone', 'Mobile Phone', 'Office Phone']].apply(lambda col: col.map(format_phone))
phones = pd.DataFrame(
    [row.dropna().tolist() + [pd.NA] * (3 - row.notna().sum()) for _, row in phones.iterrows()],
    columns=['Phone 1', 'Phone 2', 'Phone 3'],
    index=df_merged.index,
)

df_final = pd.DataFrame({
    'PRN': df_merged['Patient Identifier'],
    'First Name': df_merged['Patient First Name'].map(proper_name),
    'Last Name': df_merged['Patient Last Name'].map(proper_name),
    'Date of Birth': df_merged['Date of Birth'],
    'Age': df_merged['Age'],
    'Sex': df_merged['Sex'],
    'Appointment Type': df_merged['AppointmentType'],
    'Seen By': df_merged['SeenBy'],
    'Facility': df_merged['Facility'],
    'Address': address.replace('', pd.NA),
    'City': df_merged['City'],
    'State': df_merged['State'],
    'Postal Code': df_merged['Postal Code'],
    'Most Recent Visit Date': df_merged['_time'].dt.strftime('%m/%d/%Y'),
    'Phone 1': phones['Phone 1'],
    'Phone 2': phones['Phone 2'],
    'Phone 3': phones['Phone 3'],
    'Email Address': df_merged['Email Address'].fillna('').str.replace(r'\s', '', regex=True).str.lower().replace('', pd.NA),
})
df_final = df_final.sort_values(['Last Name', 'First Name']).reset_index(drop=True)
df_final.head()

## 13. Exportar

In [ ]:
output_name = f"{OUTPUT_PREFIX} {report_start:%m.%d.%Y}-{report_end:%m.%d.%Y}.csv"
output_path = os.path.join(APPOINTMENTS_DIR, output_name)
df_final.to_csv(output_path, index=False)
print(f"Guardado: {output_path} ({len(df_final)} pacientes)")